## Evaluation finale gelee

Ne passer `RUN_FINAL` a `True` qu'une seule fois, apres validation du manifeste versionne. Cette cellule charge les seuils figes et ne contient aucune calibration.

In [ ]:
RUN_FINAL = False

if RUN_FINAL:
    frozen = json.loads(FROZEN_CONFIG_PATH.read_text(encoding="utf-8"))
    assert frozen["version"] == "medsiglip_zero_shot_v1"
    assert frozen["model_id"] == MODEL_ID
    assert frozen["normal_prompts"] == NORMAL_PROMPTS
    assert frozen["opacity_prompts"] == OPACITY_PROMPTS

    final = selection.loc[selection["split"].eq("final")].copy()
    assert len(final) > 0, "Aucune image finale dans la selection."
    if "patient_id" in selection:
        assert set(final["patient_id"]).isdisjoint(set(dev["patient_id"]))
    if "case_id" not in final:
        final["case_id"] = [f"medsiglip_final_{index:03d}" for index in range(len(final))]

    final["study_key"] = final["image_path"].map(study_key)
    final["image_path_resolved"] = final["image_path"].map(resolve_image_path)
    final = final.merge(study_stats, on="study_key", how="left")
    assert final["image_path_resolved"].map(Path.exists).all()
    assert final["study_image_count"].notna().all()

    final_records = []
    for start in range(0, len(final), BATCH_SIZE):
        batch = final.iloc[start:start + BATCH_SIZE].copy()
        try:
            torch.cuda.synchronize()
            started = time.perf_counter()
            scores = score_batch(batch["image_path_resolved"].tolist())
            torch.cuda.synchronize()
            per_image_ms = (time.perf_counter() - started) * 1000 / len(batch)
            for (_, row), score in zip(batch.iterrows(), scores):
                record = row.to_dict()
                record.update({
                    "score_opacity": float(score),
                    "latency_ms": per_image_ms,
                    "technical_error": "",
                })
                final_records.append(record)
        except Exception as error:
            for _, row in batch.iterrows():
                record = row.to_dict()
                record.update({
                    "score_opacity": np.nan,
                    "latency_ms": np.nan,
                    "technical_error": str(error),
                })
                final_records.append(record)

    final_results = pd.DataFrame(final_records)
    final_results["predicted_class"] = classify_scores(
        final_results["score_opacity"].to_numpy(),
        float(frozen["low_threshold"]),
        float(frozen["high_threshold"]),
    )
    final_error = final_results["technical_error"].fillna("").ne("")
    final_results.loc[final_error, "predicted_class"] = "technical_error"
    final_results["model_name"] = MODEL_ID
    final_results["model_version"] = frozen["version"]
    final_results["confidence_type"] = frozen["score_type"]

    final_definitive = final_results.loc[
        final_results["expected_label"].isin(["normal", "suspected_opacity"])
        & final_results["predicted_class"].ne("technical_error")
    ].copy()
    final_positive = final_definitive["expected_label"].eq("suspected_opacity")
    final_negative = final_definitive["expected_label"].eq("normal")
    final_y = final_positive.astype(int)
    final_summary = pd.DataFrame([{
        "model_version": frozen["version"],
        "n_definitive": len(final_definitive),
        "roc_auc": roc_auc_score(final_y, final_definitive["score_opacity"]),
        "average_precision": average_precision_score(final_y, final_definitive["score_opacity"]),
        "strict_accuracy": final_definitive["predicted_class"].eq(final_definitive["expected_label"]).mean(),
        "opacity_sensitivity": final_definitive.loc[final_positive, "predicted_class"].eq("suspected_opacity").mean(),
        "opacity_to_normal_rate": final_definitive.loc[final_positive, "predicted_class"].eq("normal").mean(),
        "normal_specificity": final_definitive.loc[final_negative, "predicted_class"].eq("normal").mean(),
        "normal_to_opacity_rate": final_definitive.loc[final_negative, "predicted_class"].eq("suspected_opacity").mean(),
        "uncertain_rate": final_definitive["predicted_class"].eq("uncertain").mean(),
        "median_latency_ms": final_definitive["latency_ms"].median(),
        "technical_errors": int(final_error.sum()),
    }])

    final_results["image_path_resolved"] = final_results["image_path_resolved"].astype(str)
    final_results.to_csv(OUTPUT_DIR / "medsiglip_final_predictions.csv", index=False)
    final_summary.to_csv(OUTPUT_DIR / "medsiglip_final_summary.csv", index=False)
    display(final_summary)
    display(pd.crosstab(final_results["expected_label"], final_results["predicted_class"]))
    print("Cohorte CheXpert -1, sans accuracy:")
    display(
        final_results.loc[final_results["expected_label"].eq("uncertain"), "predicted_class"]
        .value_counts(normalize=True, dropna=False)
    )
else:
    print("Evaluation finale verrouillee. Passer RUN_FINAL a True une seule fois.")


# 04 - MedSigLIP zero-shot sur CheXpert

Objectif : evaluer `google/medsiglip-448` comme classifieur rapide de `Lung Opacity`, puis convertir son score continu en `normal`, `uncertain` ou `suspected_opacity`.

Le split `final` ne doit jamais etre utilise pour choisir les seuils. Les labels CheXpert `-1` sont analyses separement : ils ne constituent pas une verite terrain visuelle pour la classe applicative `uncertain`.

Avant de commencer, accepter les conditions MedSigLIP sur Hugging Face, ajouter `HF_TOKEN` aux secrets Kaggle et sauvegarder la selection dans `/kaggle/working/arvi_chexpert_selection.csv`.

In [ ]:
%pip install -q -U "transformers>=4.56" accelerate scikit-learn


In [ ]:
from pathlib import Path, PurePosixPath
import gc
import json
import os
import time

import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score

REPO_DIR = Path("/kaggle/working/ARVI-RX-DS-4B")
DATASET_ROOT = Path("/kaggle/input/datasets/ashery/chexpert")
SELECTION_CSV = Path("/kaggle/working/arvi_chexpert_selection.csv")
OUTPUT_DIR = Path("/kaggle/working/medsiglip_outputs")
MODEL_ID = "google/medsiglip-448"
FROZEN_CONFIG_PATH = REPO_DIR / "config" / "medsiglip_zero_shot_v1.json"
BATCH_SIZE = 8
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Active un accelerateur GPU dans Kaggle."
DEVICE = torch.device("cuda")
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(torch.cuda.get_device_name(0), DTYPE)

if not os.environ.get("HF_TOKEN"):
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")


## Charger la selection

Si la variable `selection` existe encore dans un autre notebook, l'enregistrer d'abord avec :

```python
selection.to_csv('/kaggle/working/arvi_chexpert_selection.csv', index=False)
```

In [ ]:
if not SELECTION_CSV.exists():
    candidates = list(Path("/kaggle/working").rglob("*selection*.csv"))
    raise FileNotFoundError(
        f"Selection introuvable: {SELECTION_CSV}. Candidats: {candidates}"
    )

selection = pd.read_csv(SELECTION_CSV)
if "expected_label" not in selection and "project_label" in selection:
    selection = selection.rename(columns={"project_label": "expected_label"})

required = {"split", "image_path", "expected_label"}
missing = required - set(selection.columns)
assert not missing, f"Colonnes manquantes: {sorted(missing)}"
assert set(selection["expected_label"].dropna()) <= {
    "normal", "suspected_opacity", "uncertain"
}

dev = selection.loc[selection["split"].eq("dev")].copy()
assert len(dev) > 0, "Aucune image dev dans la selection."
assert not dev["split"].eq("final").any()

if "case_id" not in dev:
    dev["case_id"] = [f"medsiglip_dev_{index:03d}" for index in range(len(dev))]

display(pd.crosstab(dev["split"], dev["expected_label"]))


In [ ]:
def study_key(path):
    parts = PurePosixPath(str(path).replace("\\", "/")).parts
    for split_name in ("train", "valid"):
        if split_name in parts:
            start = parts.index(split_name)
            return str(PurePosixPath(*parts[start:-1]))
    return str(PurePosixPath(*parts[:-1]))

def resolve_image_path(raw_path):
    direct = Path(str(raw_path))
    if direct.exists():
        return direct
    parts = PurePosixPath(str(raw_path)).parts
    if parts and parts[0] == "CheXpert-v1.0-small":
        parts = parts[1:]
    return DATASET_ROOT.joinpath(*parts)

train_csv = DATASET_ROOT / "train.csv"
train_meta = pd.read_csv(train_csv, usecols=["Path", "Frontal/Lateral"])
train_meta["study_key"] = train_meta["Path"].map(study_key)
study_stats = (
    train_meta.groupby("study_key")
    .agg(
        study_image_count=("Path", "size"),
        study_frontal_count=("Frontal/Lateral", lambda values: values.eq("Frontal").sum()),
        study_lateral_count=("Frontal/Lateral", lambda values: values.eq("Lateral").sum()),
    )
    .reset_index()
)

dev["study_key"] = dev["image_path"].map(study_key)
dev["image_path_resolved"] = dev["image_path"].map(resolve_image_path)
dev = dev.merge(study_stats, on="study_key", how="left")
assert dev["image_path_resolved"].map(Path.exists).all(), "Images introuvables."
assert dev["study_image_count"].notna().all(), "Metadonnees d'etude introuvables."

display(pd.crosstab(dev["expected_label"], dev["study_image_count"]))


## Charger MedSigLIP

Utiliser de preference une session Kaggle fraiche afin de ne pas conserver MedGemma en memoire GPU. Les scores obtenus sont des similarites relatives, pas des probabilites cliniques.

In [ ]:
from transformers import AutoModel, AutoProcessor

gc.collect()
torch.cuda.empty_cache()
processor = AutoProcessor.from_pretrained(MODEL_ID, token=os.environ["HF_TOKEN"])
model = AutoModel.from_pretrained(
    MODEL_ID,
    token=os.environ["HF_TOKEN"],
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
).to(DEVICE).eval()
print(f"Modele charge: {MODEL_ID}")


In [ ]:
NORMAL_PROMPTS = [
    "a frontal chest radiograph with clear lungs and no lung opacity",
    "a chest x-ray without focal or diffuse pulmonary opacity",
    "no visible increased density in either lung field",
]
OPACITY_PROMPTS = [
    "a frontal chest radiograph with visible lung opacity",
    "a chest x-ray showing focal or diffuse pulmonary opacity",
    "visible focal asymmetric basal or diffuse increased density in the lung fields",
]
TEXT_PROMPTS = NORMAL_PROMPTS + OPACITY_PROMPTS

def score_batch(paths):
    images = []
    for path in paths:
        with Image.open(path) as image:
            images.append(image.convert("RGB"))

    inputs = processor(
        text=TEXT_PROMPTS,
        images=images,
        padding="max_length",
        return_tensors="pt",
    )
    inputs = {name: value.to(DEVICE) for name, value in inputs.items()}
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(DTYPE)

    with torch.inference_mode():
        outputs = model(**inputs)

    logits = outputs.logits_per_image.float()
    normal_logit = logits[:, :len(NORMAL_PROMPTS)].mean(dim=1)
    opacity_logit = logits[:, len(NORMAL_PROMPTS):].mean(dim=1)
    paired_logits = torch.stack([normal_logit, opacity_logit], dim=1)
    opacity_scores = torch.softmax(paired_logits, dim=1)[:, 1]
    return opacity_scores.cpu().numpy()


In [ ]:
_ = score_batch([dev.iloc[0]["image_path_resolved"]])
torch.cuda.synchronize()

smoke = dev.head(min(BATCH_SIZE, len(dev))).copy()
started = time.perf_counter()
smoke["score_opacity"] = score_batch(smoke["image_path_resolved"].tolist())
torch.cuda.synchronize()
smoke_latency_ms = (time.perf_counter() - started) * 1000 / len(smoke)
print(f"Latence batch par image: {smoke_latency_ms:.1f} ms")
display(smoke[["case_id", "expected_label", "score_opacity"]])


## Scorer le split dev

La latence est mesuree par batch et divisee par le nombre d'images. Le premier appel de chauffe est exclu.

In [ ]:
records = []
for start in range(0, len(dev), BATCH_SIZE):
    batch = dev.iloc[start:start + BATCH_SIZE].copy()
    try:
        torch.cuda.synchronize()
        started = time.perf_counter()
        scores = score_batch(batch["image_path_resolved"].tolist())
        torch.cuda.synchronize()
        per_image_ms = (time.perf_counter() - started) * 1000 / len(batch)
        for (_, row), score in zip(batch.iterrows(), scores):
            record = row.to_dict()
            record.update({
                "score_opacity": float(score),
                "latency_ms": per_image_ms,
                "technical_error": "",
            })
            records.append(record)
    except Exception as error:
        for _, row in batch.iterrows():
            record = row.to_dict()
            record.update({
                "score_opacity": np.nan,
                "latency_ms": np.nan,
                "technical_error": str(error),
            })
            records.append(record)
    print(f"{min(start + BATCH_SIZE, len(dev))}/{len(dev)}")

raw_results = pd.DataFrame(records)
raw_results["image_path_resolved"] = raw_results["image_path_resolved"].astype(str)
raw_results.to_csv(OUTPUT_DIR / "medsiglip_dev_raw_scores.csv", index=False)
error_mask = raw_results["technical_error"].fillna("").ne("")
print(f"Erreurs techniques: {int(error_mask.sum())}")
if error_mask.any():
    display(raw_results.loc[error_mask, "technical_error"].value_counts())


## Calibrer la zone `uncertain`

Seuls `normal` et `suspected_opacity` participent au choix des seuils. Une solution admissible doit conserver au moins 85 % de sensibilite et 70 % de specificite, limiter les opacites classees `normal` a 10 %, les normaux classes `suspected_opacity` a 25 %, et garder une abstention entre 5 % et 20 %. L'accuracy stricte departage ensuite les solutions equilibrees.

In [ ]:
valid = raw_results.loc[
    raw_results["technical_error"].fillna("").eq("")
    & raw_results["expected_label"].isin(["normal", "suspected_opacity"])
].copy()
assert valid["expected_label"].nunique() == 2

y_true = valid["expected_label"].eq("suspected_opacity").astype(int)
auc = roc_auc_score(y_true, valid["score_opacity"])
average_precision = average_precision_score(y_true, valid["score_opacity"])

def classify_scores(scores, low_threshold, high_threshold):
    return np.where(
        scores >= high_threshold,
        "suspected_opacity",
        np.where(scores <= low_threshold, "normal", "uncertain"),
    )

grid = np.unique(np.r_[0.0, np.linspace(0.0, 1.0, 101), valid["score_opacity"], 1.0])
candidates = []
for low_threshold in grid:
    for high_threshold in grid[grid > low_threshold]:
        predicted = classify_scores(
            valid["score_opacity"].to_numpy(), low_threshold, high_threshold
        )
        opacity = valid["expected_label"].eq("suspected_opacity").to_numpy()
        normal = ~opacity
        sensitivity = (predicted[opacity] == "suspected_opacity").mean()
        unsafe_normal_rate = (predicted[opacity] == "normal").mean()
        if sensitivity < 0.80 or unsafe_normal_rate > 0.10:
            continue
        candidates.append({
            "low_threshold": float(low_threshold),
            "high_threshold": float(high_threshold),
            "strict_accuracy": (predicted == valid["expected_label"].to_numpy()).mean(),
            "opacity_sensitivity": sensitivity,
            "opacity_to_normal_rate": unsafe_normal_rate,
            "normal_specificity": (predicted[normal] == "normal").mean(),
            "normal_to_opacity_rate": (predicted[normal] == "suspected_opacity").mean(),
            "uncertain_rate": (predicted == "uncertain").mean(),
        })

threshold_search = pd.DataFrame(candidates)
assert not threshold_search.empty, "Aucune paire de seuils ne respecte les contraintes."
selective_candidates = threshold_search.loc[
    threshold_search["uncertain_rate"].between(0.05, 0.20)
    & threshold_search["opacity_sensitivity"].ge(0.85)
    & threshold_search["opacity_to_normal_rate"].le(0.10)
    & threshold_search["normal_specificity"].ge(0.70)
    & threshold_search["normal_to_opacity_rate"].le(0.25)
].copy()
assert not selective_candidates.empty, (
    "Aucun seuil ne respecte simultanement les contraintes de securite, "
    "specificite et abstention. Inspecter threshold_search sans choisir "
    "automatiquement un seuil desequilibre."
)
selective_candidates = selective_candidates.sort_values(
    ["strict_accuracy", "opacity_to_normal_rate", "normal_specificity", "opacity_sensitivity", "uncertain_rate"],
    ascending=[False, True, False, False, True],
)
best = selective_candidates.iloc[0]
LOW_THRESHOLD = float(best["low_threshold"])
HIGH_THRESHOLD = float(best["high_threshold"])
print({"roc_auc": auc, "average_precision": average_precision})
display(selective_candidates.head(10))


In [ ]:
results = raw_results.copy()
results["predicted_class"] = classify_scores(
    results["score_opacity"].to_numpy(), LOW_THRESHOLD, HIGH_THRESHOLD
)
results.loc[
    results["technical_error"].fillna("").ne(""), "predicted_class"
] = "technical_error"
results["model_name"] = MODEL_ID
results["confidence_type"] = "relative_similarity_uncalibrated"

definitive = results.loc[
    results["expected_label"].isin(["normal", "suspected_opacity"])
    & results["predicted_class"].ne("technical_error")
].copy()
opacity = definitive["expected_label"].eq("suspected_opacity")
normal = definitive["expected_label"].eq("normal")
summary = pd.DataFrame([{
    "model": MODEL_ID,
    "n_definitive": len(definitive),
    "roc_auc": auc,
    "average_precision": average_precision,
    "strict_accuracy": definitive["predicted_class"].eq(definitive["expected_label"]).mean(),
    "opacity_sensitivity": definitive.loc[opacity, "predicted_class"].eq("suspected_opacity").mean(),
    "opacity_to_normal_rate": definitive.loc[opacity, "predicted_class"].eq("normal").mean(),
    "normal_specificity": definitive.loc[normal, "predicted_class"].eq("normal").mean(),
    "normal_to_opacity_rate": definitive.loc[normal, "predicted_class"].eq("suspected_opacity").mean(),
    "uncertain_rate": definitive["predicted_class"].eq("uncertain").mean(),
    "coverage": definitive["predicted_class"].ne("uncertain").mean(),
    "median_latency_ms": definitive["latency_ms"].median(),
    "technical_errors": results["predicted_class"].eq("technical_error").sum(),
}])

display(summary)
display(pd.crosstab(definitive["expected_label"], definitive["predicted_class"]))


In [ ]:
ambiguous = results.loc[results["expected_label"].eq("uncertain")].copy()
print("Distribution sur les labels CheXpert -1, sans calcul d'accuracy:")
display(ambiguous["predicted_class"].value_counts(dropna=False, normalize=True))

def safe_rate(values):
    return float(values.mean()) if len(values) else np.nan

def group_metrics(name, data):
    group = data[data["expected_label"].isin(["normal", "suspected_opacity"])].copy()
    positive = group["expected_label"].eq("suspected_opacity")
    negative = group["expected_label"].eq("normal")
    return {
        "group": name,
        "n": len(group),
        "opacity_sensitivity": safe_rate(group.loc[positive, "predicted_class"].eq("suspected_opacity")),
        "opacity_to_normal_rate": safe_rate(group.loc[positive, "predicted_class"].eq("normal")),
        "normal_specificity": safe_rate(group.loc[negative, "predicted_class"].eq("normal")),
        "uncertain_rate": safe_rate(group["predicted_class"].eq("uncertain")),
    }

one_frontal = results.loc[results["study_frontal_count"].eq(1)]
frontal_only = results.loc[results["study_lateral_count"].eq(0)]
display(pd.DataFrame([
    group_metrics("all_definitive", results),
    group_metrics("one_frontal_view", one_frontal),
    group_metrics("frontal_only_studies", frontal_only),
]))


In [ ]:
threshold_config = {
    "model_id": MODEL_ID,
    "normal_prompts": NORMAL_PROMPTS,
    "opacity_prompts": OPACITY_PROMPTS,
    "low_threshold": LOW_THRESHOLD,
    "high_threshold": HIGH_THRESHOLD,
    "calibration_split": "dev_definitive_labels_only",
    "selection_constraints": {
        "min_opacity_sensitivity": 0.85,
        "max_opacity_to_normal_rate": 0.10,
        "min_normal_specificity": 0.70,
        "max_normal_to_opacity_rate": 0.25,
        "target_uncertain_rate": [0.05, 0.20],
    },
    "score_type": "relative_similarity_uncalibrated",
}

results.to_csv(OUTPUT_DIR / "medsiglip_dev_predictions.csv", index=False)
summary.to_csv(OUTPUT_DIR / "medsiglip_dev_summary.csv", index=False)
with (OUTPUT_DIR / "medsiglip_thresholds.json").open("w", encoding="utf-8") as file:
    json.dump(threshold_config, file, indent=2)

print(sorted(str(path) for path in OUTPUT_DIR.iterdir()))


## Decision apres execution

1. Comparer MedSigLIP au baseline MedGemma sur exactement les memes cas.
2. Ne pas interpreter le softmax de similarite comme une probabilite clinique.
3. Verifier en priorite les opacites classees `normal` et les etudes multi-images.
4. Ne lancer le split `final` qu'apres gel du modele, des prompts textuels et des seuils.
5. Si MedSigLIP est retenu, integrer ensuite un backend dans `src/` et garder MedGemma pour l'explication.